# How English-trained byte BPE still encodes 数

Chapter 2 · Day 2

A tokenizer can represent a character it never saw without knowing that character's meaning. We train a transparent whole-document byte BPE on English, then encode Chinese. The byte alphabet supplies coverage; learned merges supply compression. This toy omits production regex splitting and special tokens.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/dongxi_llms').exists())
sys.path.insert(0, str(ROOT / 'src'))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(909)
plt.rcParams.update({'figure.figsize': (8, 3.6), 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': False})

### Prediction

Prediction: will English-only training make 数 unknown, one token, or several byte tokens? Explain coverage separately from compression.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
from dongxi_llms.course_foundations import ByteBPE
english = ByteBPE.train(['the cat liked the garden and the cat liked the sun'] * 8, merge_count=24)
ids = english.encode('数')
print('UTF-8 bytes:', list('数'.encode('utf-8')))
print('Encoded IDs:', ids)
print('Pieces as hex:', [english.pieces[i].hex() for i in ids])
assert ids == [230, 149, 176]
assert english.decode(ids) == '数'
print('Round trip:', english.decode(ids))

### Reference explanation

Reference explanation: 数 is UTF-8 bytes E6 95 B0. The full 256-byte base alphabet already contains them, so this English-trained encoder falls back to three IDs. Individual byte pieces need not decode to meaningful characters by themselves; concatenate their bytes before decoding. These IDs are our toy's byte addresses, not Qwen or GPT-2 IDs.

For this transparent ByteBPE fixture, “数” encodes as the three UTF-8 byte IDs `[230, 149, 176]` and round-trips exactly. These IDs and counts belong to this byte vocabulary; they are not Qwen tokenizer IDs.


### Prediction

Intervention: replace the Chinese training corpus with equally frequent 数据 and 库 in separate documents.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.


### Reference solution

Run this complete existing reference after your prediction or attempt. The explanation distinguishes its actual comparison from any proposed modification.


In [ ]:
chinese = ByteBPE.train(['数据库'] * 20, merge_count=8)
print('Chinese-trained tokens:', chinese.encode('数据库'))
print('English-trained tokens:', english.encode('数据库'))
assert len(chinese.encode('数据库')) == 1

# Every box is an actual token piece, positioned by the bytes it represents.
from matplotlib.patches import Rectangle
fig, (ax, counts_ax) = plt.subplots(1, 2, figsize=(12, 4),
                                  gridspec_kw={'width_ratios': [3, 1]})
text = '数据库'
budgets = [0, 2, 5, 8]
for row, budget in enumerate(budgets):
    encoder = ByteBPE.train([text] * 20, merge_count=budget)
    token_ids = encoder.encode(text)
    assert encoder.decode(token_ids) == text
    offset = 0
    for token_id in token_ids:
        piece = encoder.pieces[token_id]
        width = len(piece)
        ax.add_patch(Rectangle((offset, row-.34), width, .68,
                               facecolor='#dde9f5', edgecolor='#3265ad'))
        ax.text(offset+width/2, row, piece.hex(' ').upper(),
                ha='center', va='center', fontsize=8, fontfamily='monospace')
        offset += width
    ax.text(9.15, row, f'{len(token_ids)} tokens', va='center', fontsize=9)
ax.set(xlim=(0, 10.5), ylim=(-.65, 3.65),
       yticks=range(4), yticklabels=[f'{n} merges' for n in budgets],
       xlabel='Byte offsets: Chinese text U+6570 U+636E U+5E93',
       title='Same text, learned byte combinations, fewer encoding atoms')
ax.invert_yaxis()
counts_ax.bar(range(len(chinese.counts)), chinese.counts, color='#3265ad')
counts_ax.set(xlabel='Learned merge index', ylabel='Winning pair count',
              title='Repeated toy corpus')
fig.tight_layout()
plt.show()


### Reference explanation

Reference: no merge crosses a document boundary in this toy, so database-as-one-piece cannot be learned from adjacency that never occurs. Increasing frequency alone cannot manufacture adjacency.

Reference explanation: the repeated nine-byte string 数据库 can become one learned piece after eight merges, while all 256 base pieces remain available. A vocabulary merge records a frequent byte composition, not an independently discovered semantic concept. Corpus frequency, merge budget and encoding rules all matter.

The current contiguous Chinese fixture starts with nine UTF-8 bytes and can merge into one piece under its displayed merge budget. Separating “数据” and “库” into different training documents is a proposed controlled modification: the unchanged cell supplies the contiguous baseline, not a measurement of that separated variant.

**Exercise modification reference.** Replace only the training list in the existing call by `["数据", "库"] * 20`, keeping `merge_count=8`; encode the unchanged text with `ByteBPE.train(["数据", "库"] * 20, merge_count=8).encode("数据库")`. Each training document has its own pair boundaries, so the learner can form “数据” and “库” pieces without learning a cross-document merge into “数据库”. The unchanged cell executes the contiguous-document baseline; the separated-document control is an unexecuted exercise.


**Saved reference preview — rerun the preceding plot cell for current inputs.**

![01 byte bpe training and encoding: reference plot 01](../figures/chapter-02/day-02-01_byte_bpe_training_and_encoding-01.png)

This fixed image comes from the CPU reference or the explicitly identified saved evidence; changing inputs does not update the preview automatically.

Evidence boundary: exact round trips and merge counts are verified for this educational encoder. Production multilingual token counts require that model's actual frozen tokenizer, including its pretokenization, normalization and added-token rules.